<a href="https://colab.research.google.com/github/kitlapp/NLP-Course/blob/main/pr_01_data_collection_and_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Import Libraries and setup Configurations

In [ ]:
# Standard library
from pathlib import Path
import re
import os

# Third-party libraries
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

pd.set_option('display.max_colwidth', None)

# 1. Data Collection & Quick Exploration

We greatly acknowledge **rohanharode07** and the **WebMD** platform for providing the dataset. The dataset can be found at:  

[WebMD Drug Reviews Dataset](https://www.kaggle.com/datasets/rohanharode07/webmd-drug-reviews-dataset)  

The dataset does not contain any copyright restrictions and can be used even for commercial purposes without asking any permission, as it can be seen in [Dataset Copyrights Rules](https://creativecommons.org/publicdomain/zero/1.0/).

## 1.1. kagglehub Mechanics

kagglehub is a Kaggle API tool used to download datasets by referencing their dataset identifier in the form "owner/dataset_name". In this case, the identifier "rohanharode07/webmd-drug-reviews-dataset" is used, where "rohanharode07" is the dataset owner and "webmd-drug-reviews-dataset" is the dataset name. This identifier is sufficient for kagglehub to locate and download the dataset automatically.

After downloading, kagglehub stores the dataset locally in a cache directory (for example on Windows: C:\Users\<user_name>\.cache\kagglehub\datasets\rohanharode07\webmd-drug-reviews-dataset\versions\1). The exact path may vary depending on the system and dataset version.

The number "1" represents the dataset version index assigned by Kaggle for that specific snapshot. Each time the dataset is updated by the owner, Kaggle assigns a new version number (e.g., 2, 3, etc.). The actual dataset files are stored inside the corresponding version folder.

The key function in this process is dataset_download, which checks whether the dataset version is already available locally. If the same version exists, it reuses the cached copy instead of downloading again. If a new version is available on Kaggle, it will download and store it in a new version folder (e.g., version 2 instead of 1), while keeping older versions locally.

In this workflow, the script uses the version returned by kagglehub.dataset_download, which is typically the latest available version at the time of download.  

For the dataset used in this case, the directory contains a single main file, webmd.csv. As a next step, the final path to the dataset file is constructed. As a final step, the webmd.csv is read to a DataFrame.

In [ ]:
# Define dataset ID according to KaggleHub format
path_to_dataset = "rohanharode07/webmd-drug-reviews-dataset"

# Download the latest available version of the dataset
# If the same version is already cached locally, it will not download again
dataset_path = kagglehub.dataset_download(path_to_dataset)

print("Dataset location:", dataset_path)

# Convert the returned path into a Path object for easier handling later
dataset_path = Path(dataset_path)

# Iterate over all items inside the dataset directory (including version folder)
print("Files found:")
for file in dataset_path.iterdir():
    print(file.name)

# Use the Path object to create the final path to the dataset
raw_filepath = dataset_path / "webmd.csv"
print("Path to raw dataset:", raw_filepath)

# Read it to a df
df_raw = pd.read_csv(raw_filepath)

## 1.2. Quick Exploration

- The dataset contains 12 columns and 362,806 observations.

- There is a clear text column that supports NLP tasks in this project. The text column is called "Reviews", and it contains patient feedback on the use of specific drugs. Drugs can be identified using the "DrugId" column, while the drug name is stored in the "Drug" column.

- The dataset also contains multiple rating columns: "EaseofUse", "Effectiveness", and "Satisfaction", which can be used as potential targets for supervised learning tasks. According to kaggle dataset link all score columns contain ratings from 1 to 5 stars.

- There are additional metadata columns that can support analysis if needed, such as Sides (reported side effects) and UsefulCount (number of users who found the review helpful).

- The remaining columns describe the review context, such as "Sex", "Date", "Age", and "Condition".

- The occupied RAM of the loaded raw dataset is 193.18Mb.

In NLP tasks, categorical metadata such as age group, sex, or condition may be used depending on the modeling approach. However, in this project the primary focus is on extracting semantic information from the "Reviews" text, with the rating columns serving as potential prediction targets, while metadata remains optional contextual information.

In [ ]:
print("Display the first 2 rows of the DataFrame:")
df_raw.head(2)

In [ ]:
print("Display general information of the DataFrame:")
df_raw.info()

In [ ]:
print("Dimensionality:", df_raw.shape)

In [ ]:
# Display how much RAM memory is occupied by the loaded dataset
def show_memory(df, name="df"):
    mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"{name} memory: {mem:.2f} MB")

show_memory(df_raw, "df_raw")

## 1.3. Medical or Business Impact

The scope of this project is to prove that free-text human reviews contain semantic information that reflects drug usage satisfaction, ease of use, and effectiveness. Instead of manually reading large volumes of reviews, modern NLP techniques can extract this information efficiently, reducing analysis time while preserving meaningful interpretation of the underlying data.

# 2. Data Preprocessing and Normalization

In [ ]:
# Copy raw dataset beforepreprocessing
df = df_raw.copy()

## 2.1. Missing Values

The dataset contains only 43 Nulls, all in "Reviews" column. This can be seen both from df_raw.info() was used above, and isna() method used here. These 43 values represent the 0.01186 % of the dataset's observations and they can be safely dropped.

In [ ]:
print("Nulls per column:", df_raw.isna().sum())

In [ ]:
# Drop any row which has at least one Null in any column
df = df.dropna()

In [ ]:
# Validation
print("Nulls per column after drop:", df.isna().sum())

## 2.2. Duplicate Values

There are 943 completely duplicated rows. These records contain identical values across all columns, indicating redundant entries. Removing them prevents artificial pollution of repeated samples and ensures uniqueness.

In [ ]:
print("Duplicates:", df.duplicated().sum())

In [ ]:
# Drop all duplicated rows keeping the first
df = df.drop_duplicates()

In [ ]:
# Validation
print("Duplicates:", df.duplicated().sum())

In [ ]:
print("Dimensionality:", df.shape)

## 2.3. A Note on Outliers

All score columns contain ordinal ratings from 1 to 5 stars. There are no outliers in the classical sense unless a score value is greater than 5, lower than 1, or negative. As will be shown later, in section 2.4., there are no outliers, but rather invalid scores.

## 2.4. Normalize all Score Columns: Keep only 5-star ratings

There are three rows where all score columns contain values greater than 5. According to the dataset description on Kaggle, these ratings are not valid within the defined 5-star scale. These records correspond to the same three indices across the dataset and should be removed to ensure consistency with the expected rating range After this drop, minimum and maximum of all score columns has to be ensured for validation reasons.

In [ ]:
print("Indexes where 'Satisfaction' rated with > 5 stars:", df[df["Satisfaction"] > 5].index.tolist())

In [ ]:
print("Indexes where 'Effectiveness' rated with > 5 stars:", df[df["Effectiveness"] > 5].index.tolist())

In [ ]:
print("Indexes where 'EaseofUse' rated with > 5 stars:", df[df["EaseofUse"] > 5].index.tolist())

In [ ]:
# Store invalid idnexes to a list
indexes_to_drop = df[df["Satisfaction"] > 5].index.tolist()

# Drop the invalid indexes
df = df.drop(index=indexes_to_drop)

# Reset index
df = df.reset_index(drop=True)

In [ ]:
# Validations
print("Indexes where 'Satisfaction' rated with > 5 stars:", df[df["Satisfaction"] > 5].index.tolist())

print("Minimum and Maximum od Score Columns:", df[["EaseofUse", "Effectiveness", "Satisfaction"]].agg(["min", "max"]))

In [ ]:
print("Dimensionality:", df.shape)

## 2.5. Scores Exploration and Normalization

### 2.5.2. Explore EaseofUse

In [ ]:
# Plotting the bar chart for the value counts of EaseofUse
score_counts = df["EaseofUse"].value_counts()
plt.figure(figsize=(10, 2))
score_counts.plot(kind='bar')
plt.title('Distribution of EaseofUse Values')
plt.xlabel('EaseofUse')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Print the statistics of EaseofUse
df["EaseofUse"].describe().to_frame().transpose()

### 2.5.3. Explore Effectiveness

In [ ]:
# Plotting the bar chart for the value counts of Effectiveness
score_counts = df["Effectiveness"].value_counts()
plt.figure(figsize=(10, 2))
score_counts.plot(kind='bar')
plt.title('Distribution of Effectiveness Values')
plt.xlabel('Effectiveness')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Print the statistics of Effectiveness
df["Effectiveness"].describe().to_frame().transpose()

### 2.5.4. Explore Satisfaction

In [ ]:
# Plotting the bar chart for the value counts of Satisfaction
score_counts = df["Satisfaction"].value_counts()
plt.figure(figsize=(10, 2))
score_counts.plot(kind='bar')
plt.title('Distribution of Satisfaction Values')
plt.xlabel('Satisfaction')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Print the statistics of Satisfaction
df["Satisfaction"].describe().to_frame().transpose()

### 2.5.5. A Note on Scores Data Types

All score columns are kept as integers because they have an ordinal interpretation. Specifically, a 1-star review demonstrates a lower score than a 2-star review and so on. If they are converted to categories, their numerical interpretation will be lost.

## 2.6. Reviews Preliminary Preprocessing

### 2.6.1. Handle Empty Reviews

The first dataset split is applied here. Empty reviews are not suitable for NLP tasks. This does not mean they do not represent valid observations, however. A dataset split seems a reasonable approach here:

1) The empty reviews dataset is kept for potential analysis --> df_non_nlp
2) The NLP dataset contains only non-empty reviews --> df_nlp

In [ ]:
# Rows where Reviews is empty or whitespace-only
empty_reviews = df[df["Reviews"].astype(str).str.strip() == ""]

In [ ]:
print("Empty Reviews Dimensionality:", empty_reviews.shape)

In [ ]:
# Split dataset into NLP-usable and empty-text rows
df_nlp = df[df["Reviews"].astype(str).str.strip() != ""].copy()
df_empty = df[df["Reviews"].astype(str).str.strip() == ""].copy()

# Reset index for clean downstream processing
df_nlp = df_nlp.reset_index(drop=True)
df_non_nlp = df_empty.reset_index(drop=True)

print("NLP dataset dimensionality:", df_nlp.shape)
print("Non NLP reviews dataset dimensionality:", df_non_nlp.shape)

### 2.6.2. Handle Punctuation Only Reviews

A custom function is used here which detects reviews that contain only non-alphanumeric characters. This function returns a boolean True or False, independently of the number of non-alphanumeric characters contained in the review. The result of this implementation shows that there are only 32 such reviews which contain only non-alphanumeric characters. Although in some cases, e.g. "!!!!", there may be a potentially strong sentiment, since this number is extremely small compared to the 320,095 NLP rows, these records are not used in the NLP dataset and are instead added to the non-NLP DataFrame, rather than dropped, as they still represent valid observations.

In [ ]:
# Function to detect punctuation-only text
def is_punct_only(text):
    text = str(text).strip()
    return bool(re.fullmatch(r"[\W_]+", text))  # keep only non-alphanumeric characters

# Create the mask
punct_only_mask = df_nlp["Reviews"].apply(is_punct_only)

# Apply the mask
punct_only_df = df_nlp[punct_only_mask]

print("Punctuation-only reviews:", len(punct_only_df))

# Move them to non-NLP dataset, rather than drop them entirely
df_non_nlp = pd.concat([df_non_nlp, punct_only_df], ignore_index=True)

# Keep only valid NLP rows
df_nlp = df_nlp[~punct_only_mask].reset_index(drop=True)

print("NLP dataset dimensionality:", df_nlp.shape)
print("Non NLP reviews dataset dimensionality:", df_non_nlp.shape)

### 2.6.3. Handle 1-Character Reviews

At this stage, and after removing punctuation-only reviews, all reviews consisting of a single character can be captured. Fortunately, there are only 11 such cases, and since they are valid records, they can be safely added to the df_non_nlp DataFrame.

In [ ]:
single_char_mask = df_nlp["Reviews"].astype(str).str.strip().str.len() == 1

single_char_df = df_nlp[single_char_mask]

print("1-character reviews:", len(single_char_df))

# Move them to non-NLP dataset, rather than drop them entirely
df_non_nlp = pd.concat([df_non_nlp, single_char_df], ignore_index=True)

# Keep only valid NLP rows
df_nlp = df_nlp[~single_char_mask].reset_index(drop=True)

print("NLP dataset dimensionality:", df_nlp.shape)
print("Non NLP reviews dataset dimensionality:", df_non_nlp.shape)

### 2.6.4. Handle Repeated Single-Character Reviews

The same applies to repeated-character reviews. A custom function is used to capture cases where a review contains repeated single characters, such as "aaa", "AAA", or mixed-case repetitions. Since these observations are considered valid records but not useful for NLP processing, they are also added to the df_non_nlp analytics DataFrame.

In [ ]:
def is_repeated_single_char_case_insensitive(text):
    text = str(text).strip().lower()
    return bool(re.fullmatch(r"(.)\1+", text))

repeated_mask = df_nlp["Reviews"].apply(is_repeated_single_char_case_insensitive)

repeated_df = df_nlp[repeated_mask]

print("Repeated single-char reviews:", len(repeated_df))

# Move them to non-NLP dataset, rather than drop them entirely
df_non_nlp = pd.concat([df_non_nlp, repeated_df], ignore_index=True)

# Keep only valid NLP rows
df_nlp = df_nlp[~repeated_mask].reset_index(drop=True)

print("NLP dataset dimensionality:", df_nlp.shape)
print("Non NLP reviews dataset dimensionality:", df_non_nlp.shape)

### 2.6.5. Handle Literal "None" Reviews

A final check is performed to identify reviews containing null-like textual values such as "None", "nan", "null", or similar placeholders. These cases are treated as missing or non-informative text entries for NLP purposes. The analysis shows that there are 77 such records. Although these observations may still correspond to valid structured data in other columns, they do not contain usable textual information and therefore cannot contribute to feature extraction. For consistency with the previous cleaning steps, these records are removed from the NLP dataset and added to the non-NLP DataFrame.

In [ ]:
null_like_values = {"none", "nan", "null", "n/a", "na"}

mask_null_like = df_nlp["Reviews"].astype(str).str.strip().str.lower().isin(null_like_values)

null_like_df = df_nlp[mask_null_like]

print("Null-like reviews:", len(null_like_df))

# Move them to non-NLP dataset, rather than drop them entirely
df_non_nlp = pd.concat([df_non_nlp, null_like_df], ignore_index=True)

# Keep only valid NLP rows
df_nlp = df_nlp[~mask_null_like].reset_index(drop=True)

print("NLP dataset dimensionality:", df_nlp.shape)
print("Non NLP reviews dataset dimensionality:", df_non_nlp.shape)

## 2.8. Normalize Texts

For this part, the professor’s notes are used entirely, as they are considered a safe point for text normalization.

In [ ]:
# Define some variables
SIZE = 2

### 2.8.1. Lower all Letters

In [ ]:
def lowercase_text(text):
    # Convert text to lowercase.
    return str(text).lower()

# Apply lowercase function to all summaries
df_nlp["Reviews"] = df_nlp["Reviews"].apply(lowercase_text)

In [ ]:
# Display a few examples to verify the transformation
print("After lowercase transformation:")
rand_idxs = np.random.randint(0, len(df_nlp["Reviews"]), size=SIZE)
for idx in rand_idxs:
    print(f"Score: {df_nlp["Satisfaction"].iloc[idx]} - Summary: {df_nlp["Reviews"].iloc[idx]}")

### 2.8.2. Replace URLs

In [ ]:
PRINT_COUNTER = {"count": 0}
# Define a regex pattern to identify URLs in the text
url_pattern = r"(?:https?|ftp)://[^\s/$.?#].[^\s]*"

def replace_urls(text):
    """
    Replace URLs in the text with the token 'URL'.
    Prints before and after if a replacement occurs.
    """
    text_str = str(text)
    replaced_text = re.sub(url_pattern, 'URL', text_str)

    if replaced_text != text_str:
        if PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
            print(f"Before: {text_str}")
            print(f"After:  {replaced_text}\n")

        PRINT_COUNTER["count"] += 1  # Added by me to handle huge outputs

    return replaced_text

# Apply URL replacement to all summaries
df_nlp["Reviews"] = df_nlp["Reviews"].apply(replace_urls)

### 2.8.3. Handle Emogis and Emoticon

In [ ]:
# re.compile will compile the regex pattern into a regex object, necessary for
# efficient pattern matching. This creates a reusable pattern object that can be
# used multiple times without recompiling the pattern each time, improving performance.
# u stands for Unicode
emoji_pattern = re.compile("["

    # Emoticons (e.g., 😀😁😂🤣😃😄😅😆)
    u"\U0001F600-\U0001F64F"

    # Symbols & pictographs (e.g., 🔥🎉💡📦📱)
    u"\U0001F300-\U0001F5FF"

    # Transport & map symbols (e.g., 🚗✈️🚀🚉)
    u"\U0001F680-\U0001F6FF"

    # Flags (e.g., 🇺🇸🇬🇧🇨🇦 — these are pairs of regional indicators)
    u"\U0001F1E0-\U0001F1FF"

    # Dingbats (e.g., ✂️✈️✉️⚽)
    u"\u2700-\u27BF"

    # Supplemental Symbols & Pictographs (e.g., 🤖🥰🧠🦾)
    u"\U0001F900-\U0001F9FF"

    # Symbols & Pictographs Extended-A (e.g., 🪄🪅🪨)
    u"\U0001FA70-\U0001FAFF"

    # Miscellaneous symbols (e.g., ☀️☁️☂️⚡)
    u"\u2600-\u26FF"

    "]+", flags=re.UNICODE)

In [ ]:
# This pattern will match common text-based emoticons that aren't covered by the emoji Unicode ranges
# These emoticons are made up of regular ASCII characters like colons, parentheses, etc.
# Examples include:
# :) - happy face
# :( - sad face
# :D - laughing face
# ;) - winking face
emoticon_pattern = re.compile(r'(:\)|:\(|:D|:P|;\)|:-\)|:-D|:-P|:\'\(|:\||:\*)')

In [ ]:
PRINT_COUNTER = {"count": 0}
def remove_and_print(text):
    """
    Remove emojis/emoticons from text.
    Print only the first 5 modified examples to avoid cluttering logs.
    """

    # Check whether the text contains emojis or emoticons
    if emoji_pattern.search(text) or emoticon_pattern.search(text):

        if PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
            print(f"Before: {text}")

        # Remove emojis and emoticons
        text = emoji_pattern.sub('', text)
        text = emoticon_pattern.sub('', text)

        if PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
            print(f"After: {text}")
            print()

        PRINT_COUNTER["count"] += 1  # Added by me to handle huge outputs

    return text


# Apply transformation to the full Series
df_nlp["Reviews"] = df_nlp["Reviews"].apply(remove_and_print)

### 2.8.4. Replace Usernames

Usernames and email addresses were identified using regular expressions and anonymized by replacing them with a generic token (“USER”) to ensure privacy preservation. This includes both standalone @usernames and email-based identifiers. No additional explicit personally identifiable information (PII), such as phone numbers or physical addresses, was identified using the applied preprocessing rules during dataset inspection.

In [ ]:
PRINT_COUNTER = {"count": 0}
def replace_usernames(text):
    """
    Replace email addresses and @usernames with 'USER'.
    Print only the first 5 replacements.
    """

    original = str(text)
    updated = original

    # Replace email addresses
    updated = re.sub(
        r'\b[\w\.-]+@[\w\.-]+\.\w+\b',
        'USER',
        updated
    )

    # Replace standalone @usernames
    updated = re.sub(
        r'(?:(?<=^)|(?<=[\s.,;!?]))@\w+\b',
        'USER',
        updated
    )

    if updated != original:

        if PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
            print(f"Before: {original}")
            print(f"After:  {updated}\n")

        PRINT_COUNTER["count"] += 1  # Added by me to handle huge outputs

    return updated


# Apply transformation
df_nlp["Reviews"] = df_nlp["Reviews"].apply(replace_usernames)

### 2.8.5. Remove Non-Alphabets

In [ ]:
PRINT_COUNTER = {"count": 0}
def clean_text(text, keep_punct=False):
    """
    Clean and normalize text for NLP classification tasks.

    Parameters:
    - text (str): The input text to be cleaned.
    - keep_punct (bool):
        If True, retains key punctuation (. ! ?) which may carry emotional or contextual weight.
        If False, removes all non-alphabetic characters for simpler lexical analysis.

    Returns:
    - str: The cleaned text string, lowercased and stripped of unwanted characters.

    This function is designed for flexibility across different NLP tasks like sentiment analysis,
    topic classification, or spam detection. It handles:
    - Lowercasing text for normalization
    - Removing or preserving select punctuation
    - Removing digits, symbols, and special characters
    - Reducing multiple spaces to a single space
    - Optionally printing changes for debugging or logging

    When to use `keep_punct=True`:
    - Sentiment analysis: punctuation (e.g., "!", "?") can reflect strong emotion
    - Social media or informal text: expressive punctuation often carries signal
    - Sarcasm, emphasis, or tone-sensitive tasks

    When to use `keep_punct=False`:
    - Topic classification or document clustering: punctuation rarely adds value
    - Preprocessing for bag-of-words, TF-IDF, or topic modeling
    - When punctuation is inconsistent or noisy (e.g., OCR scans, scraped data)
    """

    # Convert input to string (safe handling)
    original = str(text)

    if keep_punct:
        # Keep only lowercase letters, spaces, and select punctuation (. ! ?)
        # Useful for capturing tone/sentiment
        cleaned = re.sub(r"[^a-z\s.!?]", "", original)
    else:
        # Keep only lowercase letters and spaces; remove all punctuation and symbols
        cleaned = re.sub(r"[^a-z\s]", "", original)

    # Normalize whitespace (collapse multiple spaces to one, strip leading/trailing)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    # Optional: print before and after if a change occurred
    if original != cleaned:

        if PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
            print(f"Before: {text}")
            print(f"After:  {cleaned}\n")

        PRINT_COUNTER["count"] += 1  # Added by me to handle huge outputs

    return cleaned

# Apply non-alphabet removal to all summaries
df_nlp["Reviews"] = df_nlp["Reviews"].apply(lambda x: clean_text(x, keep_punct=True))

### 2.8.6. Remove Consecutive Letters

In [ ]:
PRINT_COUNTER = {"count": 0}
def remove_consecutive_letters(text, max_repeat=2):
    """
    Normalize elongated words by limiting repeated characters.

    In informal or emotional text (e.g., reviews, tweets), users often repeat letters
    to add emphasis: "sooooo good", "loooove it", "greeaaat".

    This function reduces any character repeated more than `max_repeat` times
    to exactly `max_repeat` occurrences (default: 2), preserving emphasis without bloating vocabulary.

    Parameters:
    - text (str): The input text
    - max_repeat (int): The maximum allowed repetitions for any character

    Returns:
    - str: Text with repeated characters normalized
    """
    text_str = str(text)
    pattern = r'(\w)\1{' + str(max_repeat) + r',}'
    cleaned = re.sub(pattern, r'\1' * max_repeat, text_str)

    # Print only if changes were made
    if cleaned != text_str:

        if PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
            print(f"Before: {text_str}")
            print(f"After:  {cleaned}\n")

        PRINT_COUNTER["count"] += 1  # Added by me to handle huge outputs

    return cleaned

# Apply consecutive letter removal to all summaries
df_nlp["Reviews"] = df_nlp["Reviews"].apply(lambda x: remove_consecutive_letters(x, max_repeat=2))

### 2.8.7. Remove Short Words

In [ ]:
PRINT_COUNTER = {"count": 0}

def remove_short_words(text, min_length=3, preserve_words=None):
    """
    Remove short words from text based on a minimum length threshold.

    Parameters:
    - text (str): The input text
    - min_length (int): Minimum word length to keep (default = 3)
    - preserve_words (set or list): Optional set of short but important words to keep (e.g., {'no', 'not'})

    Returns:
    - str: Text with short words removed, except for preserved ones

    Notes:
    - Use with care in sentiment analysis. Important short words like 'no', 'not', 'bad' may affect meaning.
    - Best used after stopword removal or on very noisy text.
    """
    preserve = set(preserve_words or [])
    words = str(text).split()
    filtered = [word for word in words if len(word) >= min_length or word.lower() in preserve]
    result = ' '.join(filtered)

    if result != text:

        if PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
            print(f"Before: {text}")
            print(f"After:  {result}\n")

        PRINT_COUNTER["count"] += 1  # Added by me to handle huge outputs

    return result

# Apply short word removal to all summaries
df_nlp["Reviews"] = df_nlp["Reviews"].apply(lambda x: remove_short_words(x, min_length=3, preserve_words={'no', 'not'}))

### 2.8.8. Removing Stopwords

In [ ]:
# Download stopwords if not already downloaded
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
print("Sample stopwords:", list(stopwords.words('english'))[:10])

In [ ]:
# Define stopwords but keep critical ones like "not"
base_stopwords = set(stopwords.words('english'))
preserve = {'no', 'not', 'nor', 'never'}
custom_stopwords = base_stopwords - preserve

In [ ]:
PRINT_COUNTER = {"count": 0}

def remove_stopwords(text):
    """
    Remove stopwords from text, preserving key negation words.

    This function uses a customized stopword list that retains important
    short words like 'not', 'no', 'nor', and 'never' which carry significant
    meaning in tasks like sentiment analysis.

    Parameters:
    - text (str): Lowercased input text

    Returns:
    - str: Text with stopwords removed, but critical negation words preserved
    """
    words = str(text).split()
    filtered = [word for word in words if word not in custom_stopwords]
    result = ' '.join(filtered)

    if result != text:
        if PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
            print(f"Before: {text}")
            print(f"After:  {result}\n")

        PRINT_COUNTER["count"] += 1  # Added by me to handle huge outputs

    return result

# Apply remove_stopwords
df_nlp["Reviews"] = df_nlp["Reviews"].apply(remove_stopwords)

### 2.8.9. A Note on Lemmatization

Lemmatization is beneficial for classical NLP methods such as One-Hot Encoding, Bag of Words, and TF-IDF, because these approaches treat words as discrete tokens. By reducing different word forms to a common base form, lemmatization decreases vocabulary size and reduces sparsity in the feature space, which can improve model performance and generalization. However, modern NLP methods such as transformers and other deep learning-based models are designed to learn contextual representations of words directly from raw text and can effectively handle different word forms without requiring lemmatization. For this reason, the dataset is split into two versions to support both types of approaches: one dataset that applies lemmatization for classical NLP pipelines, and another that preserves the original preprocessed text for deep learning and transformer-based models.

#### 2.8.9.1. Non Lemmatized Dataset

In [ ]:
df_non_lem = df_nlp.copy()

#### 2.8.9.2. Lemmatized Dataset

In [ ]:
df_lem = df_nlp.copy()

In [ ]:
# Download required NLTK resources
nltk.download('wordnet')  # Download WordNet, a lexical database of English words
nltk.download('omw-1.4')  # WordNet Lemmas sometimes need this, which is a mapping of WordNet lemmas to their Part of Speech (POS) tags.
nltk.download('averaged_perceptron_tagger_eng')  # Download English POS tagger

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

In [ ]:
# POS mapping function
# POS tags can be: ADJ (adjective), ADV (adverb), NOUN (noun), VERB (verb), etc
def get_wordnet_pos(tag):
    # Determine the WordNet POS tag based on the first letter of the input tag
    if tag.startswith('J'):
        return wordnet.ADJ  # Adjective
    elif tag.startswith('V'):
        return wordnet.VERB  # Verb
    elif tag.startswith('N'):
        return wordnet.NOUN  # Noun
    elif tag.startswith('R'):
        return wordnet.ADV  # Adverb
    else:
        return wordnet.NOUN  # Default to Noun if no match

PRINT_COUNTER = {"count": 0}

def lemmatize_text(text):
    """
    Lemmatize text using WordNet lemmatizer with POS tagging.

    This version prints each change along with the POS tag of the changed word.
    """
    # Convert the input text to a string to ensure compatibility
    original_text = str(text)
    # Split the text into individual words
    words = original_text.split()
    # Obtain Part of Speech (POS) tags for each word
    pos_tags = pos_tag(words)

    # Initialize lists to store lemmatized words and any changes
    lemmatized_words = []
    changes = []

    # Iterate over each word and its POS tag
    for word, tag in pos_tags:
        # Map the POS tag to a WordNet POS tag
        wn_tag = get_wordnet_pos(tag)
        # Lemmatize the word using the mapped POS tag
        lemma = lemmatizer.lemmatize(word, wn_tag)

        # Check if the lemmatized word is different from the original
        if lemma != word:
            # Record the change if a difference is found
            changes.append((word, lemma, tag))
        # Add the lemmatized word to the list
        lemmatized_words.append(lemma)

    # Join the lemmatized words back into a single string
    result = ' '.join(lemmatized_words)

    # Print only if there were changes
    if changes and PRINT_COUNTER["count"] < SIZE:  # Added by me to handle huge outputs
      print(f"\nOriginal: {original_text}")
      print(f"Lemmatized: {result}")
      for original, lemma, pos in changes:
          print(f"  - {original} → {lemma}  (POS: {pos})")

      PRINT_COUNTER["count"] += 1  # Added by me to handle huge outputs
    return result

# Apply lemmatization to all summaries
df_lem["Reviews"] = df_lem["Reviews"].apply(lemmatize_text)

## 2.9. Word Cloud Implementation

The term “side effect” appears as one of the most frequent words in both positive and negative review groups, for both lemmatized and non-lemmatized data. This shows that side effects is a very common topic in drug reviews and it is mentioned almost in all cases, independent of sentiment. Patients talk about side effects whether their overall experience is good or bad, which means it is an important factor in how they describe a medication. In real life this is expected, because side effects are a key concern when people use drugs. However, frequency alone does not show sentiment, it only shows importance of the topic inside the dataset.

### 2.9.1. Positive Sentiments (Lemmatized)

In [ ]:
# Filter dataset: keep only reviews with high satisfaction (rating >= 4)
filtered_summaries = df_lem[df_lem["Satisfaction"] >= 4]

# Combine all review texts into one large string
# astype(str) ensures all values are treated as text
all_summaries = " ".join(filtered_summaries["Reviews"].astype(str))

# Generate a word cloud from the combined text
# Parameters control image size and background color
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate(all_summaries)

# Plot the generated word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')  # Hide axes for better visualization
plt.show()

### 2.9.2. Negative Sentiments (Lemmatized)

In [ ]:
filtered_summaries = df_lem[df_lem["Satisfaction"] < 3]

all_summaries = " ".join(filtered_summaries["Reviews"].astype(str))

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_summaries)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

### 2.9.3. Positive Sentiments (Non Lemmatized)

In [ ]:
filtered_summaries = df_non_lem[df_non_lem["Satisfaction"] >= 4]

all_summaries = " ".join(filtered_summaries["Reviews"].astype(str))

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_summaries)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

### 2.9.4. Negative Sentiments (Non Lemmatized)

In [ ]:
filtered_summaries = df_non_lem[df_non_lem["Satisfaction"] < 3]

all_summaries = " ".join(filtered_summaries["Reviews"].astype(str))

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_summaries)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

## 2.10. Tokenization

In [ ]:
# Download required NLTK tokenization resources
# "punkt" is the standard sentence tokenizer model
nltk.download("punkt")

# "punkt_tab" is an additional tokenizer resource used in newer NLTK setups
# It may be required depending on the environment/version
nltk.download("punkt_tab")

In [ ]:
# Tokenize raw (non-lemmatized) reviews into word-level tokens
# astype(str) ensures all values are treated as strings to avoid errors
df_non_lem["tokens"] = df_non_lem["Reviews"].astype(str).apply(word_tokenize)

# Tokenize lemmatized reviews in the same way
# This allows consistent comparison between raw and processed text
df_lem["tokens"] = df_lem["Reviews"].astype(str).apply(word_tokenize)

In [ ]:
print("Display the first 2 rows of the tokens:")
df_non_lem.head(2)

In [ ]:
print("Display the first 2 rows of the tokens:")
df_lem.head(2)

## 2.11. Feature Engineer Score_Avg

At this stage, a new feature-engineered column is created. Since the dataset contains three different rating dimensions (Satisfaction, Effectiveness, and Ease of Use), it is reasonable to create a fourth score column representing their arithmetic mean.

This additional feature is not created because it is necessarily required in the subsequent stages of this project. Instead, it is introduced for the following reasons:

1. To demonstrate an understanding of feature engineering techniques and their application in real-world machine learning workflows.

2. To provide a unified score representing the overall patient experience, avoiding the need to select a single rating column as the definitive target variable for future NLP experiments.

3. To simulate real-world data science practices, where different stakeholders, teams, and departments often require different versions of cleaned and feature-engineered datasets depending on their objectives, analytical requirements, and domain expertise.

In [ ]:
# Columns containing numerical evaluation scores
score_cols = ["Satisfaction", "Effectiveness", "EaseofUse"]

# Compute average score per row for lemmatized dataset
# axis=1 means row-wise mean across selected columns
# round(..., 2) keeps 2 decimal precision
df_lem["Scores_Avg"] = round(df_lem[score_cols].mean(axis=1), 2)

# Compute average score per row for non-lemmatized dataset
# Ensures both representations are comparable under same metric
df_non_lem["Scores_Avg"] = round(df_non_lem[score_cols].mean(axis=1), 2)

In [ ]:
print("Display the first 2 rows of the tokens:")
df_non_lem.sample(2)

In [ ]:
print("Display the first 2 rows of the tokens:")
df_lem.sample(2)

# 3. Validations and Local Save

The most important DataFrames produced in this stage are:

1) The non-NLP analytics DataFrame, which is stored for potential future analysis outside the scope of this project. It is saved to demonstrate a structured and reproducible workflow → "df_analytics_final.parquet". This dataset contains reviews that were excluded from NLP processing because they do not provide meaningful textual content for language-based methods. However, they may still be useful for alternative analytical purposes.

2) The NLP-ready lemmatized DataFrame, which is suitable for classical NLP methods such as Bag of Words, TF-IDF, and traditional machine learning models → "df_lem_final.parquet".

3) The NLP-ready non-lemmatized DataFrame, which preserves the original cleaned text and is more appropriate for modern approaches such as deep learning models, transformer-based architectures, and embedding-based methods → "df_non_lem_final.parquet".

In [ ]:
# Select columns to compare between lemmatized and non-lemmatized datasets
# Excludes text and token columns, since they are expected to differ
cols_to_compare = [col for col in df_lem.columns if col not in ["Reviews", "tokens"]]

# Create aligned subsets for fair comparison
# reset_index ensures row indices match for element-wise comparison
df_lem_subset = df_lem[cols_to_compare].reset_index(drop=True)
df_non_lem_subset = df_non_lem[cols_to_compare].reset_index(drop=True)

# Element-wise comparison between both dataframes
# .all().all() checks if all values are identical across all rows and columns
(df_lem_subset == df_non_lem_subset).all().all()

In [ ]:
# df_non_nlp.to_parquet("df_analytics_final.parquet", index=False)
# df_lem.to_parquet("df_lem_final.parquet", index=False)
# df_non_lem.to_parquet("df_non_lem_final.parquet", index=False)

# 4. Cloud Sharing

The above datasets are stored locally and, since they exceed GitHub's file size limits for practical repository management, they are shared through a personal Google Drive folder as follows:

[Final Cleaned Datasets](https://drive.google.com/drive/folders/1spDsOxCQ2BARxbTrWTqTPRZEXoxvlDtl?usp=sharing)